# 03 · Part 3: scaling

**Question.** On SIFT, how does the cost of reaching recall 0.95 grow with the number of base vectors N, for exact search, IVF-Flat and HNSW? The hypothesis is in `hypothesis.md`, committed before this notebook first ran. The next cell refuses to continue unless that commit exists.

**Setup.** The base set is the first N SIFT vectors, at three sizes; the queries and the query halves stay the same, and ground truth is recomputed for each N. One thing changes at a time: N. HNSW keeps M = 16 and efConstruction = 200. IVF-Flat sets nlist = round(4√N), so the average list holds √N / 4 vectors. Each index is swept and its operating point at 0.95 chosen exactly as in Part 2. Choices are recorded in `design_note.md`.

In [ ]:
%run ./00_core.ipynb

In [ ]:
require_committed_hypothesis("Part 3")

## Configurations

In [ ]:
def scaling_configs(n, config=CONFIG):
    """(index kind, build settings, search values) for the two indexes at base size n."""
    nlist = scaling_nlist(n)
    return [("ivf_flat", dict(nlist=nlist), [p for p in config["nprobe"] if p <= nlist]),
            ("hnsw", dict(M=config["scaling_M"], efC=config["hnsw_efC"]), config["ef_search"])]


pd.DataFrame([dict(n_base=f"{n:,}", index=kind, build=json.dumps(params), search=f"{SEARCH_PARAM[kind]} up to {max(values)}")
              for n in CONFIG["scaling_sizes"] for kind, params, values in scaling_configs(n)])

In [ ]:
%%ipytest -qq

def test_full_mode_uses_the_assignment_sizes():
    assert FULL["scaling_sizes"] == [100_000, 300_000, 1_000_000]


@pytest.mark.parametrize("config", [FULL, QUICK], ids=["full", "quick"])
def test_only_nlist_changes_with_n(config):
    first, *rest = [scaling_configs(n, config) for n in config["scaling_sizes"]]
    for runs in rest:
        assert runs[1] == first[1]                                   # HNSW: identical at every size
        assert runs[0][1]["nlist"] > first[0][1]["nlist"]            # IVF-Flat: more lists as N grows
    for n in config["scaling_sizes"]:
        assert max(scaling_configs(n, config)[0][2]) <= scaling_nlist(n)

## Run
SIFT is loaded once at the largest size; each smaller base set is its prefix. Each size's rows are written as soon as they are measured.

In [ ]:
largest = load_dataset("sift", max(CONFIG["scaling_sizes"]))
for n in CONFIG["scaling_sizes"]:
    started = time.perf_counter()
    data = Dataset("sift", largest.base[:n], largest.queries, None, None)
    gt = ground_truth("sift", data.base, data.queries)
    rows = benchmark_exact(3, str(n), data, gt)
    for kind, params, values in scaling_configs(n):
        rows += benchmark_index(3, str(n), data, gt, kind, params, values)
    write_rows(rows, part=3, section=str(n))
    print(f"N={n:,}: {len(rows)} rows in {(time.perf_counter() - started) / 60:.1f} min")
del largest, data, gt

## Cost of recall 0.95 against N
Left: report-half latency at each index's operating point for recall 0.95. Right: distance computations per query at the same points, which is the work behind the latency. Both axes are logarithmic, so a power law N^b is a straight line of slope b. The label at each line's end is its fitted slope: about 1 means the cost grows like N, 0.5 like √N, and close to 0 like log N.

In [ ]:
results = read_results()
figure, tables = part3_assets(results)
plt.show()

## Tables
- **Cost at each size:** the setting chosen for 0.95 on the tune half, with its report-half recall, QPS, latency, distance computations, build time and bytes per vector.
- **Growth:** fitted slopes for latency and for distance computations, how much each grew over the measured range, and the Q4 inputs. Latency at 150K comes from the fit inside the measured range; latency at 10M extrapolates it 10× beyond the largest size. Memory is bytes per vector at the largest size × N.

In [ ]:
for name, table in tables.items():
    print(name)
    display(table)

In [ ]:
%%ipytest -qq

scaling = scaling_table(results)


def test_every_size_has_all_three_methods():
    methods = scaling.groupby("n_base")["index"].agg(set)
    assert all(found == {"exact", "ivf_flat", "hnsw"} for found in methods)
    assert sorted(scaling["n_base"].unique()) == CONFIG["scaling_sizes"]


def test_both_indexes_reach_095_at_every_size():
    assert (scaling["setting"] != "not reached").all()


def test_exact_search_computes_every_distance():
    exact = scaling[scaling["index"] == "exact"]
    assert (exact["distances_per_query"] == exact["n_base"]).all()